# Deploy Lakehouse Agent

Deploy the Lakehouse Agent to AgentCore Runtime.

## Prerequisites

- ✅ Run `04-deploy-gateway.ipynb` first
- ✅ Gateway ARN saved to SSM
- ✅ Docker installed and running

## What This Notebook Does

1. Deploys Lakehouse Agent to AgentCore Runtime
2. Configures Gateway integration
3. Saves Agent Runtime ARN to SSM

## Next Notebook

- **06-test-deployment.ipynb**

In [5]:
import sys
sys.path.insert(0, '.')  # Add current directory to path
from pathlib import Path
from aws_session_utils import get_aws_session

# Get validated AWS session with SSO support
session, region, account_id = get_aws_session()

# Initialize AWS clients
ssm_client = session.client('ssm', region_name=region)

# Load Gateway ARN from SSM
try:
    gateway_arn = ssm_client.get_parameter(
        Name='/app/lakehouse-agent/gateway-arn'
    )['Parameter']['Value']
    print('✅ Setup complete')
    print(f'   Region: {region}')
    print(f'   Gateway ARN: {gateway_arn}')
except ssm_client.exceptions.ParameterNotFound:
    print('❌ Gateway ARN not found in SSM')
    print('   Please run 04-deploy-gateway.ipynb first')
    gateway_arn = None

🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Setup complete
   Region: us-east-1
   Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:gateway/lakehouse-gateway-1ekjecoowq


## Step 1: Deploy Lakehouse Agent

In [6]:
import subprocess

# Run deploy_lakehouse_agent.py with --yes flag to skip interactive prompts
result = subprocess.run(
    ['python', 'deploy_lakehouse_agent.py', '--yes'],
    cwd='lakehouse-agent',
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr)
else:
    print('\n✅ Lakehouse Agent deployed!')
    print('\n📋 Configuration automatically saved to SSM')

Lakehouse Data Agent Deployment to AgentCore Runtime
🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Configuration loaded
   Region: us-east-1
   Account: XXXXXXXXXXXX

🔍 Loading configuration from SSM Parameter Store...
   ✅ Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:gateway/lakehouse-gateway-1ekjecoowq
   ✅ Cognito configured

🔍 Validating configuration...
✅ Configuration validated

📋 Configuration:
   Region: us-east-1
   Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:gateway/lakehouse-gateway-1ekjecoowq

✅ Auto-confirming deployment (--yes flag provided)

Step 1: Creating IAM Role
Creating IAM role: AgentCoreRuntimeRole-lakehous

## Step 2: Verify Agent Deployment

The deploy_lakehouse_agent.py script automatically saves the Runtime ARN to SSM.
Run this cell to verify the deployment.

In [7]:
# Verify Agent configuration in SSM
print("Verifying Agent Runtime configuration in SSM...\n")

parameters_to_check = [
    '/app/lakehouse-agent/agent-runtime-arn',
    '/app/lakehouse-agent/agent-runtime-id',
    '/app/lakehouse-agent/agent-name',
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response['Parameter']['Value']
        print(f'✅ {param_name}')
        print(f'   Value: {value}')
    except ssm_client.exceptions.ParameterNotFound:
        print(f'⚠️  {param_name} - NOT FOUND (optional)')
        if 'agent-runtime-arn' in param_name:
            all_found = False
    except Exception as e:
        print(f'⚠️  {param_name} - ERROR: {e}')

if all_found:
    print('\n✅ Agent Runtime configuration verified in SSM!')
else:
    print('\n❌ Agent Runtime ARN missing in SSM.')
    print('    The deploy_lakehouse_agent.py script should have saved this automatically.')
    print('    Check the deployment output for errors.')

Verifying Agent Runtime configuration in SSM...

✅ /app/lakehouse-agent/agent-runtime-arn
   Value: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/lakehouse_agent-HBEJQxHyGT
✅ /app/lakehouse-agent/agent-runtime-id
   Value: lakehouse_agent-HBEJQxHyGT
✅ /app/lakehouse-agent/agent-name
   Value: lakehouse_agent

✅ Agent Runtime configuration verified in SSM!


## Summary

✅ **Lakehouse Agent Deployment Complete!**

The Agent Runtime has been deployed and configuration saved to SSM Parameter Store.

**Next Steps:**
Run **06-test-deployment.ipynb** to test the complete system